In [1]:
import pprint
import radiate as rd
import numpy as np
import polars as pl

rd.random.seed(123)

In [2]:
def compute(x: float) -> float:
    return 4.0 * x**3 - 3.0 * x**2 + x


inputs = []
answers = []

x = -1.0
for _ in range(20):
    x += 0.1
    inputs.append([x])
    answers.append([compute(x)])

X = np.array(inputs, dtype=np.float32)  # (N, 1)
Y = np.array(answers, dtype=np.float32)  # (N, 1)

Xb = np.concatenate([X, np.ones((X.shape[0], 1), dtype=np.float32)], axis=1)


def fit(weights: list[np.ndarray]) -> float:
    W1 = weights[0].reshape((8, 2))
    W2 = weights[1].reshape((8, 8))
    W3 = weights[2].reshape((1, 8))

    h1 = Xb @ W1.T
    h1 = np.maximum(0, h1)

    h2 = h1 @ W2
    h2 = np.tanh(h2)

    yhat = h2 @ W3.T

    return float(np.mean((yhat - Y) ** 2, dtype=np.float32))

In [3]:
collector = rd.MetricCollector()

engine = (
    rd.Engine.float(
        shape=[16, 64, 8],
        init_range=(-1.0, 1.0),
        bounds=(-3.0, 3.0),
        use_numpy=True,
        dtype=rd.Float32,
    )
    .fitness(fit)
    .subscribe(collector)
    .minimizing()
    .select(rd.Select.boltzmann(temp=4.0))
    .alters(rd.Cross.blend(0.7, 0.4), rd.Mutate.gaussian(0.3))
    .limit(rd.Limit.score(0.01), rd.Limit.generations(500))
)

result = engine.run(log=True)

2026-06-09T01:19:24.946092Z  INFO Epoch 1    | Score:   1.1841 | Time: 2.01ms
2026-06-09T01:19:24.946967Z  INFO Epoch 2    | Score:   0.9343 | Time: 2.84ms
2026-06-09T01:19:24.947827Z  INFO Epoch 3    | Score:   0.8793 | Time: 3.66ms
2026-06-09T01:19:24.948741Z  INFO Epoch 4    | Score:   0.8793 | Time: 4.54ms
2026-06-09T01:19:24.949731Z  INFO Epoch 5    | Score:   0.8697 | Time: 5.49ms
2026-06-09T01:19:24.950611Z  INFO Epoch 6    | Score:   0.7182 | Time: 6.33ms
2026-06-09T01:19:24.951511Z  INFO Epoch 7    | Score:   0.7182 | Time: 7.20ms
2026-06-09T01:19:24.952369Z  INFO Epoch 8    | Score:   0.5674 | Time: 8.03ms
2026-06-09T01:19:24.953219Z  INFO Epoch 9    | Score:   0.5674 | Time: 8.85ms
2026-06-09T01:19:24.954071Z  INFO Epoch 10   | Score:   0.5674 | Time: 9.67ms
2026-06-09T01:19:24.955318Z  INFO Epoch 11   | Score:   0.5085 | Time: 10.88ms
2026-06-09T01:19:24.956195Z  INFO Epoch 12   | Score:   0.3981 | Time: 11.72ms
2026-06-09T01:19:24.957058Z  INFO Epoch 13   | Score:   0.3797

In [4]:
# collector.plot(
#     "count.species",
# )

In [5]:
metrics = result.metrics()
df = metrics.to_polars()
df

name,last,sum,mean,stddev,var,skew,min,max,count,time_sum,time_mean,time_stddev,time_min,time_max,time_var,generation,update_count,tags
str,f64,f64,f64,f64,f64,f64,f64,f64,i64,duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],i64,i64,list[str]
"""count.evaluation""",80.0,40105.0,80.049904,0.9206,0.847505,0.0,80.0,100.0,501,null,null,null,null,null,null,499,1,"[""statistic""]"
"""step.evaluate.time""",0.0007,0.373514,0.000374,0.000376,1.4142e-7,0.0,8.3000e-8,0.001567,1000,373514µs,373µs,376µs,0µs,1566µs,0µs,499,2,"[""time"", ""step""]"
"""selector.roulette""",20.0,10000.0,20.0,0.0,0.0,0.0,20.0,20.0,500,null,null,null,null,null,null,499,1,"[""selector"", ""statistic""]"
"""selector.roulette.time""",0.000001,0.000711,0.000001,7.0548e-7,4.9770e-13,0.0,0.000001,0.00001,500,710µs,1µs,0µs,1µs,9µs,0µs,499,1,"[""selector"", ""time""]"
"""selector.boltzmann""",80.0,40000.0,80.0,0.0,0.0,0.0,80.0,80.0,500,null,null,null,null,null,null,499,1,"[""selector"", ""statistic""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""time""",0.000812,0.432579,0.000865,0.000088,7.8178e-9,0.0,0.00081,0.002007,500,432579µs,865µs,88µs,809µs,2007µs,0µs,499,1,"[""time""]"
"""index""",500.0,500.0,500.0,0.0,0.0,NaN,500.0,500.0,1,null,null,null,null,null,null,0,1,"[""statistic""]"
"""score.improvement""",1.0,18.0,1.0,0.0,0.0,0.0,1.0,1.0,18,null,null,null,null,null,null,167,1,"[""statistic"", ""score""]"


In [6]:
print(result.metrics().dashboard())

for metric in metrics.values_by_tag(rd.Tag.DERIVED):
    print(metric)

print()
pprint.pprint(metrics["rate.carryover"].to_dict())

[30 metrics, 14020 updates]
----- Metrics ----- (30 :: 14020) 
  carryover: 0.166  diversity: 0.960  unique_members: 93  unique_scores: 93  improvements: 18  iter_time(mean): 865.158µs

== Distributions ==
Name                       | Type   | Mean       | Min        | Max        | N      | Total        | StdDev     | Skew       | Kurt      
---------------------------------------------------------------------------------------------------------------------------------------
age                        | dist   | 0.890      | 0.000      | 12.000     | 100    | 0.089        | 2.856      | 4.475      | 19.595    
scores                     | dist   | 2.442      | 0.208      | 16.559     | 100    | 0.244        | 2.538      | 3.400      | 18.825    
size.genome                | dist   | 88.000     | 88.000     | 88.000     | 100    | 8.800        | 0.000      | 0.000      | 0.000     


== Statistics ==
Name                       | Type   | Mean       | Min        | Max        | N      | T

In [7]:
df = collector.to_polars(lazy=False)

df = (
    df.filter(
        (pl.col("update_count") > 1) & (~pl.col("tags").list.contains("distribution"))
    )
    .select("name", "generation", "update_count", "tags")
    .group_by("name")
    .agg(
        pl.col("update_count").sum().alias("update_count"),
    )
    .sort("update_count", descending=False)
)

df

name,update_count
str,i64
"""count.evaluation""",2
"""step.evaluate.time""",1000


In [8]:
df = collector.to_polars(lazy=False)

In [9]:
df.filter(pl.col("name") == "gaussian_mutator")

name,last,sum,mean,stddev,var,skew,min,max,count,time_sum,time_mean,time_stddev,time_min,time_max,time_var,generation,update_count,tags
str,f64,f64,f64,f64,f64,f64,f64,f64,i64,duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],i64,i64,list[str]
